> **Repository version.** This notebook was cleaned from the frozen manuscript workflow:
> outputs were removed and local absolute paths were replaced by the repository `PROJECT_DIR`.
> The statistical logic was not intentionally changed.


In [ ]:
# Repository path setup
from pathlib import Path
import os

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
PROJECT_DIR = Path(
    os.environ.get("CGN_PROJECT_DIR", REPO_ROOT / "workspace")
).expanduser().resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "reproduction" / "results").mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Working data/results directory:", PROJECT_DIR)


In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

big_path = str(PROJECT_DIR / 'GSE294965_processed_data.h5ad')

adata = sc.read_h5ad(
    big_path,
    backed="r"
)

obs = adata.obs

In [ ]:
roi_mask_all = (
    (obs["is_in_glom"].to_numpy() == True) |
    (obs["is_in_periglom"].to_numpy() == True)
)

roi_global_idx_all = np.flatnonzero(
    roi_mask_all
)

roi_obs = obs.iloc[
    roi_global_idx_all
].copy()

roi_obs["roi_id"] = (
    roi_obs["glom_ID"]
    .astype(str)
)

# 去掉非具体肾小球的 other_ 区域
valid_roi = ~roi_obs["roi_id"].str.startswith(
    "other_"
)

roi_obs = roi_obs.loc[
    valid_roi
].copy()

roi_global_idx = roi_global_idx_all[
    valid_roi.to_numpy()
]

print("ROI-associated cells:",
      len(roi_obs))

print("Unique ROI:",
      roi_obs["roi_id"].nunique())

print(
    roi_obs.groupby(
        "Disease",
        observed=True
    )["roi_id"].nunique()
)

In [ ]:
# 真正存在glomerular cells的glom_ID
true_glom_ids = set(
    obs.loc[
        obs["is_in_glom"].to_numpy() == True,
        "glom_ID"
    ].astype(str)
)

# 所有periglomerular里的glom_ID
peri_ids = set(
    obs.loc[
        obs["is_in_periglom"].to_numpy() == True,
        "glom_ID"
    ].astype(str)
)

extra_peri_ids = peri_ids - true_glom_ids

print("True glomeruli:", len(true_glom_ids))
print("Periglomerular IDs:", len(peri_ids))
print("Extra peri-only IDs:", len(extra_peri_ids))

print("\nExamples:")
print(list(extra_peri_ids)[:20])

In [ ]:
glom_id_str = obs["glom_ID"].astype(str)

roi_mask_all = (
    (
        (obs["is_in_glom"].to_numpy() == True) |
        (obs["is_in_periglom"].to_numpy() == True)
    )
    &
    glom_id_str.isin(true_glom_ids).to_numpy()
)

roi_global_idx = np.flatnonzero(
    roi_mask_all
)

roi_obs = obs.iloc[
    roi_global_idx
].copy()

roi_obs["roi_id"] = (
    roi_obs["glom_ID"].astype(str)
)

print("ROI-associated cells:", len(roi_obs))
print("Unique ROI:", roi_obs["roi_id"].nunique())

print(
    roi_obs.groupby(
        "Disease",
        observed=True
    )["roi_id"].nunique()
)

In [ ]:
# 所有落入扩张glomerular polygon的细胞
roi_cells = obs[
    obs["is_in_polygon"].to_numpy() == True
].copy()

roi_cells["roi_id"] = (
    roi_cells["polygon_flags"].astype(str)
)

# 去掉同时属于多个expanded polygons的重叠区域
roi_cells_unique = roi_cells[
    ~roi_cells["roi_id"].str.contains(
        ",",
        regex=False
    )
].copy()

print(
    "ROI-associated cells:",
    len(roi_cells_unique)
)

print(
    "Unique ROI:",
    roi_cells_unique["roi_id"].nunique()
)

print("\nROIs by disease:")
print(
    roi_cells_unique
    .groupby(
        "Disease",
        observed=True
    )["roi_id"]
    .nunique()
)

print("\nCompartments included:")
print(
    roi_cells_unique[
        ["is_in_glom", "is_in_periglom"]
    ].sum()
)

In [ ]:
# 找到这些ROI细胞在原始AnnData里的行位置
roi_global_idx = obs.index.get_indexer(
    roi_cells_unique.index
)

print("Any missing indices:",
      np.any(roi_global_idx < 0))

In [ ]:
cat = pd.Categorical(
    roi_cells_unique["roi_id"].astype(str)
)

codes = cat.codes
roi_ids = cat.categories.astype(str)

n_roi = len(roi_ids)
n_cells = len(codes)

cells_per_roi = np.bincount(
    codes,
    minlength=n_roi
)

weights = 1.0 / cells_per_roi[codes]

G_roi = sp.csr_matrix(
    (
        weights,
        (
            codes,
            np.arange(n_cells)
        )
    ),
    shape=(n_roi, n_cells)
)

X_cells = adata.layers["counts"][
    roi_global_idx,
    :
]

X_roi = G_roi @ X_cells

print("ROI expression shape:",
      X_roi.shape)

print("Sparse:",
      sp.issparse(X_roi))

In [ ]:
roi_meta = (
    roi_cells_unique
    .groupby(
        "roi_id",
        observed=True
    )
    .agg(
        Disease=("Disease", "first"),
        Patient_Sample_ID=("Patient_Sample_ID", "first"),
        Biopsy_ID=("Biopsy_ID", "first")
    )
)

roi_meta = roi_meta.loc[
    roi_ids
].copy()

roi_meta["n_cells"] = cells_per_roi

print(roi_meta.shape)
roi_meta.head()

In [ ]:
roi_ad = ad.AnnData(
    X=X_roi.tocsr(),
    obs=roi_meta
)

roi_ad.var_names = (
    adata.var_names.astype(str).copy()
)

roi_ad.layers["mean_counts"] = (
    roi_ad.X.copy()
)

print(roi_ad)

In [ ]:
sc.pp.normalize_total(
    roi_ad,
    target_sum=10000
)

sc.pp.log1p(roi_ad)

sc.tl.pca(
    roi_ad,
    svd_solver="arpack"
)

roi_ad.obs["PC1_raw"] = (
    roi_ad.obsm["X_pca"][:, 0]
)

roi_ad.obs.groupby(
    "Disease",
    observed=True
)["PC1_raw"].describe()

In [ ]:
roi_ad.obs["PC1_crescent"] = roi_ad.obs["PC1_raw"].copy()

print(
    roi_ad.obs.groupby(
        "Disease",
        observed=True
    )["PC1_crescent"].median()
)

In [ ]:
print(
    "PC1 variance explained:",
    roi_ad.uns["pca"]["variance_ratio"][0]
)

In [ ]:
from scipy.stats import spearmanr

sc.pp.neighbors(roi_ad)

sc.tl.diffmap(roi_ad)

control_idx = np.flatnonzero(
    roi_ad.obs["Disease"].to_numpy() == "Cntrl"
)

rng = np.random.default_rng(2026)

root = int(
    rng.choice(control_idx)
)

print("DPT root:", root)
print(
    "Root ROI:",
    roi_ad.obs_names[root]
)

roi_ad.uns["iroot"] = root

sc.tl.dpt(roi_ad)

In [ ]:
print(
    "dpt_pseudotime" in roi_ad.obs.columns
)

print(
    roi_ad.obs["dpt_pseudotime"].describe()
)

In [ ]:
rho, p = spearmanr(
    roi_ad.obs["PC1_crescent"],
    roi_ad.obs["dpt_pseudotime"]
)

print("PC1 vs DPT Spearman rho =", rho)
print("P value =", p)

In [ ]:
roi_ad.write_h5ad(
    str(PROJECT_DIR / 'roi_782_PC1_primary.h5ad')
)

print("Saved")

In [ ]:
roi_cells_unique["macro_type"] = "Other"

roi_cells_unique.loc[
    roi_cells_unique["celltype_l1"].isin(["MAC", "Mono"]),
    "macro_type"
] = "Myeloid"

roi_cells_unique.loc[
    roi_cells_unique["celltype_l1"] == "EC",
    "macro_type"
] = "Vascular"

roi_cells_unique.loc[
    roi_cells_unique["celltype_l1"].isin(
        ["FIB", "Fibrotic MC"]
    ),
    "macro_type"
] = "Stromal"

print(
    roi_cells_unique["macro_type"].value_counts()
)

In [ ]:
pc_ranges = (
    roi_ad.obs[
        roi_ad.obs["Disease"].isin(
            ["ANCA", "SLE", "GBM"]
        )
    ]
    .groupby(
        "Disease",
        observed=True
    )["PC1_crescent"]
    .agg(["min", "max"])
)

display(pc_ranges)

pc_low = pc_ranges["min"].max()
pc_high = pc_ranges["max"].min()

print(
    "Common PC1 support:",
    pc_low,
    "to",
    pc_high
)

In [ ]:
roi_common = roi_ad.obs[
    roi_ad.obs["Disease"].isin(
        ["ANCA", "SLE", "GBM"]
    )
    &
    roi_ad.obs["PC1_crescent"].between(
        pc_low,
        pc_high
    )
].copy()

print("ROI counts:")
print(
    roi_common["Disease"].value_counts()
)

print("\nPatient counts:")
print(
    roi_common
    .groupby(
        "Disease",
        observed=True
    )["Patient_Sample_ID"]
    .nunique()
)

In [ ]:
from scipy.spatial import cKDTree
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from patsy import bs
from statsmodels.stats.multitest import multipletests

common_roi_ids = roi_common.index.astype(str)

spatial_roi_cells = roi_cells_unique[
    roi_cells_unique["roi_id"].astype(str).isin(
        common_roi_ids
    )
].copy()

print("Cells in common-support ROIs:",
      len(spatial_roi_cells))

print("Unique ROIs:",
      spatial_roi_cells["roi_id"].nunique())

print(
    spatial_roi_cells.groupby(
        "Disease",
        observed=True
    )["roi_id"].nunique()
)

In [ ]:
def neighbor_enrichment_one_roi(
    d,
    source,
    target,
    k=6,
    min_source=3,
    min_target=3
):
    
    src_mask = (
        d["macro_type"].to_numpy()
        == source
    )
    
    tgt_mask = (
        d["macro_type"].to_numpy()
        == target
    )

    n = len(d)
    n_src = src_mask.sum()
    n_tgt = tgt_mask.sum()

    if (
        n < 5 or
        n_src < min_source or
        n_tgt < min_target
    ):
        return np.nan

    xy = d[
        ["x_centroid", "y_centroid"]
    ].to_numpy()

    tree = cKDTree(xy)

    kk = min(k + 1, n)

    _, nbr_idx = tree.query(
        xy[src_mask],
        k=kk
    )

    # 去掉自己
    nbr_idx = nbr_idx[:, 1:]

    observed = (
        tgt_mask[nbr_idx]
        .mean()
    )

    expected = (
        n_tgt / (n - 1)
    )

    eps = 1e-6

    return np.log2(
        (observed + eps) /
        (expected + eps)
    )

In [ ]:
pairs = [
    ("Myeloid", "Vascular"),
    ("Myeloid", "Stromal"),
    ("Stromal", "Vascular")
]

rows = []

for roi_id, d in spatial_roi_cells.groupby(
    "roi_id",
    observed=True
):

    row = {
        "roi_id": roi_id
    }

    for source, target in pairs:

        name = (
            f"{source}_to_{target}"
        )

        row[name] = (
            neighbor_enrichment_one_roi(
                d,
                source,
                target,
                k=6
            )
        )

    rows.append(row)

pc1_neighbor = pd.DataFrame(
    rows
).set_index("roi_id")

In [ ]:
pc1_neighbor["Disease"] = (
    roi_ad.obs.loc[
        pc1_neighbor.index,
        "Disease"
    ].astype(str)
)

pc1_neighbor["Patient"] = (
    roi_ad.obs.loc[
        pc1_neighbor.index,
        "Patient_Sample_ID"
    ].astype(str)
)

pc1_neighbor["PC1"] = (
    roi_ad.obs.loc[
        pc1_neighbor.index,
        "PC1_crescent"
    ]
)

print(pc1_neighbor.shape)

print(
    pc1_neighbor[
        [
            "Myeloid_to_Vascular",
            "Myeloid_to_Stromal",
            "Stromal_to_Vascular"
        ]
    ].notna().sum()
)

In [ ]:
neighbor_vars = [
    "Myeloid_to_Vascular",
    "Myeloid_to_Stromal",
    "Stromal_to_Vascular"
]

pc1_neighbor_results = []

for var in neighbor_vars:

    d = pc1_neighbor.dropna(
        subset=[var]
    ).copy()

    d["Disease"] = pd.Categorical(
        d["Disease"],
        categories=[
            "ANCA",
            "SLE",
            "GBM"
        ]
    )

    # patient-balanced weighting
    n_roi = d.groupby(
        "Patient",
        observed=True
    )["Patient"].transform("size")

    d["patient_weight"] = (
        1.0 / n_roi
    )

    fit = smf.wls(
        (
            f"{var} ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d["patient_weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": d["Patient"]
        }
    )

    terms = [
        term for term in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]

    R = np.zeros(
        (
            len(terms),
            len(fit.params)
        )
    )

    for i, term in enumerate(terms):

        R[
            i,
            fit.params.index.get_loc(term)
        ] = 1

    wt = fit.wald_test(
        R,
        scalar=True
    )

    pc1_neighbor_results.append([
        var,
        float(wt.pvalue),
        np.linalg.matrix_rank(
            fit.model.exog
        ),
        fit.model.exog.shape[1]
    ])

pc1_neighbor_results = pd.DataFrame(
    pc1_neighbor_results,
    columns=[
        "neighbor_relation",
        "trajectory_pvalue",
        "matrix_rank",
        "n_columns"
    ]
)

pc1_neighbor_results["FDR"] = (
    multipletests(
        pc1_neighbor_results[
            "trajectory_pvalue"
        ],
        method="fdr_bh"
    )[1]
)

pc1_neighbor_results.sort_values(
    "FDR"
)

In [ ]:
pc1_neighbor_results[
    pc1_neighbor_results["matrix_rank"]
    !=
    pc1_neighbor_results["n_columns"]
]

In [ ]:
pc1_neighbor_pairwise = []

disease_pairs = [
    ("ANCA", "SLE"),
    ("ANCA", "GBM"),
    ("SLE", "GBM")
]

for var in neighbor_vars:

    for d1, d2 in disease_pairs:

        d = pc1_neighbor[
            pc1_neighbor["Disease"].isin(
                [d1, d2]
            )
        ].dropna(
            subset=[var]
        ).copy()

        d["Disease"] = pd.Categorical(
            d["Disease"],
            categories=[d1, d2]
        )

        n_roi = d.groupby(
            "Patient",
            observed=True
        )["Patient"].transform("size")

        d["patient_weight"] = (
            1.0 / n_roi
        )

        fit = smf.wls(
            (
                f"{var} ~ "
                "bs(PC1, df=3, degree=3, "
                "include_intercept=False) "
                "* C(Disease)"
            ),
            data=d,
            weights=d["patient_weight"]
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": d["Patient"]
            }
        )

        terms = [
            term for term
            in fit.params.index
            if ":" in term
            and "C(Disease)" in term
            and "bs(PC1" in term
        ]

        R = np.zeros(
            (
                len(terms),
                len(fit.params)
            )
        )

        for i, term in enumerate(terms):

            R[
                i,
                fit.params.index.get_loc(term)
            ] = 1

        wt = fit.wald_test(
            R,
            scalar=True
        )

        pc1_neighbor_pairwise.append([
            var,
            f"{d1}_vs_{d2}",
            float(wt.pvalue)
        ])

pc1_neighbor_pairwise = pd.DataFrame(
    pc1_neighbor_pairwise,
    columns=[
        "neighbor_relation",
        "comparison",
        "pvalue"
    ]
)

pc1_neighbor_pairwise["FDR"] = (
    pc1_neighbor_pairwise
    .groupby(
        "comparison",
        observed=True
    )["pvalue"]
    .transform(
        lambda x:
        multipletests(
            x,
            method="fdr_bh"
        )[1]
    )
)

pc1_neighbor_pairwise.sort_values(
    ["comparison", "FDR"]
)

In [ ]:
k_values = [4, 6, 8, 10]

pc1_k_rows = []

for k in k_values:

    temp_rows = []

    for roi_id, d in spatial_roi_cells.groupby(
        "roi_id",
        observed=True
    ):

        value = neighbor_enrichment_one_roi(
            d,
            source="Myeloid",
            target="Stromal",
            k=k
        )

        temp_rows.append([
            roi_id,
            value
        ])

    temp = pd.DataFrame(
        temp_rows,
        columns=[
            "roi_id",
            "Myeloid_to_Stromal"
        ]
    ).set_index("roi_id")

    temp["Disease"] = (
        roi_ad.obs.loc[
            temp.index,
            "Disease"
        ].astype(str)
    )

    temp["Patient"] = (
        roi_ad.obs.loc[
            temp.index,
            "Patient_Sample_ID"
        ].astype(str)
    )

    temp["PC1"] = (
        roi_ad.obs.loc[
            temp.index,
            "PC1_crescent"
        ]
    )

    # 只做目前最重要的 SLE vs GBM
    d = temp[
        temp["Disease"].isin(
            ["SLE", "GBM"]
        )
    ].dropna(
        subset=["Myeloid_to_Stromal"]
    ).copy()

    d["Disease"] = pd.Categorical(
        d["Disease"],
        categories=["SLE", "GBM"]
    )

    n_roi = d.groupby(
        "Patient",
        observed=True
    )["Patient"].transform("size")

    d["patient_weight"] = 1.0 / n_roi

    fit = smf.wls(
        (
            "Myeloid_to_Stromal ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d["patient_weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": d["Patient"]
        }
    )

    terms = [
        term for term in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]

    R = np.zeros(
        (
            len(terms),
            len(fit.params)
        )
    )

    for i, term in enumerate(terms):
        R[
            i,
            fit.params.index.get_loc(term)
        ] = 1

    wt = fit.wald_test(
        R,
        scalar=True
    )

    pc1_k_rows.append([
        k,
        float(wt.pvalue),
        np.linalg.matrix_rank(
            fit.model.exog
        ),
        fit.model.exog.shape[1],
        d["Patient"].nunique(),
        d[d["Disease"] == "GBM"]["Patient"].nunique()
    ])

pc1_k_sensitivity = pd.DataFrame(
    pc1_k_rows,
    columns=[
        "k",
        "pvalue_SLE_vs_GBM",
        "matrix_rank",
        "n_columns",
        "n_patients_total",
        "n_GBM_patients"
    ]
)

pc1_k_sensitivity

In [ ]:
gbm_patients_pc1 = (
    pc1_neighbor.loc[
        pc1_neighbor["Disease"] == "GBM",
        "Patient"
    ]
    .astype(str)
    .unique()
)

print(gbm_patients_pc1)
print("n =", len(gbm_patients_pc1))

In [ ]:
pc1_loo_rows = []

for dropped in gbm_patients_pc1:

    d = pc1_neighbor[
        pc1_neighbor["Disease"].isin(
            ["SLE", "GBM"]
        )
    ].dropna(
        subset=["Myeloid_to_Stromal"]
    ).copy()

    d = d[
        d["Patient"].astype(str)
        != str(dropped)
    ].copy()

    d["Disease"] = pd.Categorical(
        d["Disease"],
        categories=["SLE", "GBM"]
    )

    n_roi = d.groupby(
        "Patient",
        observed=True
    )["Patient"].transform("size")

    d["patient_weight"] = 1.0 / n_roi

    fit = smf.wls(
        (
            "Myeloid_to_Stromal ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d["patient_weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": d["Patient"]
        }
    )

    terms = [
        term for term in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]

    R = np.zeros(
        (
            len(terms),
            len(fit.params)
        )
    )

    for i, term in enumerate(terms):
        R[
            i,
            fit.params.index.get_loc(term)
        ] = 1

    wt = fit.wald_test(
        R,
        scalar=True
    )

    pc1_loo_rows.append([
        dropped,
        float(wt.pvalue),
        np.linalg.matrix_rank(
            fit.model.exog
        ),
        fit.model.exog.shape[1]
    ])

pc1_spatial_loo = pd.DataFrame(
    pc1_loo_rows,
    columns=[
        "dropped_GBM_patient",
        "pvalue",
        "matrix_rank",
        "n_columns"
    ]
)

pc1_spatial_loo

In [ ]:
print(
    pc1_spatial_loo["pvalue"].describe()
)

print(
    "Fraction P < 0.05:",
    (
        pc1_spatial_loo["pvalue"] < 0.05
    ).mean()
)

In [ ]:
pc1_spatial_loo[
    pc1_spatial_loo["matrix_rank"]
    != pc1_spatial_loo["n_columns"]
]

In [ ]:
pc1_k_sensitivity.to_csv(
    str(PROJECT_DIR / 'day4_PC1_myeloid_stromal_k_sensitivity.csv'),
    index=False
)

pc1_spatial_loo.to_csv(
    str(PROJECT_DIR / 'day4_PC1_myeloid_stromal_GBM_LOO.csv'),
    index=False
)

pc1_neighbor_results.to_csv(
    str(PROJECT_DIR / 'day4_PC1_neighbor_global.csv'),
    index=False
)

pc1_neighbor_pairwise.to_csv(
    str(PROJECT_DIR / 'day4_PC1_neighbor_pairwise.csv'),
    index=False
)

In [ ]:
pc1_neighbor.to_csv(
    str(PROJECT_DIR / 'day4_PC1_neighbor_raw.csv')
)

print(pc1_neighbor.shape)
print("Saved")